In [1]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.prompts import PromptTemplate

from langchain_classic.chains import RetrievalQA

c:\Users\NATHAN\Langchain\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
## Read the pdfs from the folder
loader = PyPDFDirectoryLoader("./us_census")

documents = loader.load()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)

final_documents = text_splitter.split_documents(documents)
final_documents[0]

Document(metadata={'producer': 'Adobe PDF Library 17.0', 'creator': 'Adobe InDesign 18.2 (Windows)', 'creationdate': '2023-09-09T07:52:17-04:00', 'author': 'U.S. Census Bureau', 'keywords': 'acsbr-015', 'moddate': '2023-09-12T14:44:47+01:00', 'title': 'Health Insurance Coverage Status and Type by Geography: 2021 and 2022', 'trapped': '/false', 'source': 'us_census\\acsbr-015.pdf', 'total_pages': 18, 'page': 0, 'page_label': '1'}, page_content='Health Insurance Coverage Status and Type \nby Geography: 2021 and 2022\nAmerican Community Survey Briefs\nACSBR-015\nIssued September 2023\nDouglas Conway and Breauna Branch\nINTRODUCTION\nDemographic shifts as well as economic and govern-\nment policy changes can affect people’s access to \nhealth coverage. For example, between 2021 and 2022, \nthe labor market continued to improve, which may \nhave affected private coverage in the United States \nduring that time.\n1 Public policy changes included \nthe renewal of the Public Health Emergency, 

In [3]:
len(final_documents)

316

In [4]:
## Embedding using HuggingFace
huggingface_embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={'device': 'cpu'},
    encode_kwargs={'normalize_embeddings': True}
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 795.34it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [5]:
import numpy as np
np.array(huggingface_embeddings.embed_query(final_documents[0].page_content)).shape

(384,)

In [6]:
#Vector Store Creation
vectorstore = FAISS.from_documents(final_documents[:120], huggingface_embeddings)

In [7]:
# Query using Similarity Search
query = "What is HEALTH INSURANCE COVERAGE?"
relevant_documents = vectorstore.similarity_search(query)

print(relevant_documents[0].page_content)

2 U.S. Census Bureau
WHAT IS HEALTH INSURANCE COVERAGE?
This brief presents state-level estimates of health insurance coverage 
using data from the American Community Survey (ACS). The  
U.S. Census Bureau conducts the ACS throughout the year; the 
survey asks respondents to report their coverage at the time of 
interview. The resulting measure of health insurance coverage, 
therefore, reflects an annual average of current comprehensive 
health insurance coverage status.* This uninsured rate measures a 
different concept than the measure based on the Current Population 
Survey Annual Social and Economic Supplement (CPS ASEC). 
For reporting purposes, the ACS broadly classifies health insurance 
coverage as private insurance or public insurance. The ACS defines 
private health insurance as a plan provided through an employer 
or a union, coverage purchased directly by an individual from an 
insurance company or through an exchange (such as healthcare.


In [8]:
retriver = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 3})
print(retriver)

tags=['FAISS', 'HuggingFaceEmbeddings'] vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x00000290E565CA90> search_kwargs={'k': 3}


In [9]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [10]:
os.environ["HUGGINGFACEHUB_API_TOKEN"] = os.getenv("HUGGINGFACE_API_TOKEN")

In [11]:
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint

llm = HuggingFaceEndpoint(
    repo_id="Qwen/Qwen3-32B",
    temperature=0.7,
)

model = ChatHuggingFace(llm=llm)

response = model.invoke("Why do parrots talk?")
print(response.content)

<think>
Okay, so the user is asking why parrots talk. Let me start by recalling what I know about parrots. They're part of the Psittaciformes order, right? I know they're known for their ability to mimic human speech, but why do they do that?

First, maybe I should think about evolution. Parrots have a complex vocal system. Other birds can mimic sounds too, like some songbirds, but parrots are especially good at it. Maybe it's an evolutionary trait that helps them in the wild. For example, in the wild, parrots might mimic other bird calls or environmental sounds to communicate with their flock, warn of danger, or find mates. But how does that translate to talking with humans?

I remember reading that parrots have a structure called the syrinx, which allows them to produce a wide range of sounds. Humans have vocal cords, but the syrinx is different. The syrinx is located at the base of the trachea and allows for more complex sound modulation. So parrots can learn to replicate sounds the

In [12]:
prompt_template = """
Use the following piece of context to answer the question asked.
Please try to provide the answer based on the context.

{context}
Question:{question}

Helpful Answers:
"""

In [13]:
prompt = PromptTemplate(template=prompt_template, input_variables=["context","question"])


In [14]:
retrievalQA = RetrievalQA.from_chain_type(
    llm=model,
    chain_type="stuff",
    retriever=retriver,
    return_source_documents=True,
    chain_type_kwargs={"prompt": prompt}
)


In [15]:
query="""DIFFERENCES IN THE
UNINSURED RATE BY STATE
IN 2022"""

In [16]:
# Call the QA chain with our query.
result = retrievalQA.invoke({"query": query})
print(result['result'])

<think>
Okay, let's tackle this question about the differences in the uninsured rate by state in 2022. The user provided a context from the U.S. Census Bureau, and I need to base my answer on that.

First, I need to read through the context carefully. Let me start by breaking down the information given. The context mentions several states and how their uninsured rates changed, particularly focusing on Medicaid expansion and how public and private coverage rates influenced these changes. 

The first paragraph talks about seven states where the decrease in the uninsured rate was due to increases in private coverage: Florida, Kansas, Mississippi, North Carolina, Ohio, South Carolina, and Texas. Then there are seven other states (Alabama, California, Georgia, Illinois, Indiana, Michigan, Oklahoma) where the decrease was due to public coverage increases. Three states (Missouri, New York, Virginia) saw shifts from private to public coverage contributing to the decline.

Next, there's a menti